# MariaDB Connection Check

이 노트북은 `.env` 값을 읽어 MariaDB 연결을 확인하고,
테이블 목록과 샘플 데이터를 조회하는 가장 단순한 예제입니다.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

db_type = os.getenv("DB_TYPE")
if db_type != "mariadb":
    raise ValueError(f"DB_TYPE must be 'mariadb'. current: {db_type}")


In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", "3306")),
    database=os.getenv("DB_NAME"),
)

engine = create_engine(url)
engine


In [ ]:
with engine.connect() as conn:
    version = conn.execute(text("SELECT VERSION() AS version")).scalar()

print(version)

In [ ]:
with engine.connect() as conn:
    tables = conn.execute(text("SHOW TABLES")).fetchall()

tables

아래 셀에서 조회할 테이블명을 넣어 샘플 데이터를 확인합니다.

`content` 컬럼에 HTML이 들어있는 경우를 대비해, 태그를 제거하고 텍스트만 추출하는 전처리 함수를 먼저 정의합니다.

In [ ]:
from bs4 import BeautifulSoup


def extract_text_from_html(html: str) -> str:
    if not html:
        return ""

    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style"]):
        tag.decompose()

    text = soup.get_text(separator=" ", strip=True)
    return " ".join(text.split())


In [ ]:
table_name = "post_history"
# sample_limit = 5

In [ ]:
query = text(f"SELECT * FROM `{table_name}`")

with engine.connect() as conn:
    rows = conn.execute(query).fetchall()

rows

샘플 행에서 `content` 컬럼의 HTML을 텍스트로 변환합니다.

In [ ]:
if not rows:
    raise ValueError("조회된 데이터가 없습니다.")

first_row = rows[0]
row_dict = dict(first_row._mapping)

html_content = row_dict.get("content", "")
clean_text = extract_text_from_html(html_content)

print(clean_text[:3000])

여러 행을 RAG용 문서 형태로 바꾸려면 아래처럼 사용하면 됩니다.

In [ ]:
from langchain_core.documents import Document

docs = []

for row in rows:
    item = dict(row._mapping)
    cleaned_content = extract_text_from_html(item.get("content", ""))
    docs.append(
        Document(
            page_content=cleaned_content,
            metadata={"id": item.get("id")},
        )
    )

docs[:2]

이제 `docs`를 벡터스토어에 넣고, 질문과 가장 관련 있는 문서를 검색합니다.

In [16]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model=os.getenv("EMBEDDING_MODEL", "text-embedding-3-small"))
vectorstore = FAISS.from_documents(docs, embeddings)


In [32]:
# question = "설비 소모품 구매 절차를 알려줘"
question = "메뉴얼 총 몇개있어?"
retrieved_docs = vectorstore.similarity_search(question, k=3)

retrieved_docs

[Document(id='94cc6146-1589-4d79-97ef-c1621c57d102', metadata={'title': '메뉴변경 및 타 사용자 수정 테스트(수정자: 김성진)'}, page_content='메뉴변경 및 타 사용자 수정 테스트 개정번호 \u200b 시행일자 개정내용 작성자 비고 0 20XX. XX. XX 신규 \u200b 수정 내용 등 \u200b ■ 중분류 업무명 : ■ 소분류 업무명 : 목적 ■ 담당자 : ㅇㅇㅇ / ■ 팀장 : ㅇㅇㅇ NO 업무절차 산출자료/팀명 \u200b 입력자료/팀명 자료(서비스)명 수신부서 결재경로 \u200b 자료(서비스)명 \u200b 발신부서 1 \u200b\u200b\u200b \u200b\u200b 2 \u200b3\u200b \u200b4 \u200b5 \u200b6 \u200b7 \u200b\u200b'),
 Document(id='947d0811-cc69-4062-a9e5-327a8afe1ea5', metadata={'title': '메뉴변경 및 타 사용자 수정 테스트(수정자: 김성진)'}, page_content='메뉴변경 및 타 사용자 수정 테스트 개정번호 \u200b 시행일자 개정내용 작성자 비고 0 20XX. XX. XX 신규 \u200b 수정 내용 등 \u200b ■ 중분류 업무명 : ■ 소분류 업무명 : 목적 ■ 담당자 : ㅇㅇㅇ / ■ 팀장 : ㅇㅇㅇ NO 업무절차 산출자료/팀명 \u200b 입력자료/팀명 자료(서비스)명 수신부서 결재경로 \u200b 자료(서비스)명 \u200b 발신부서 1 \u200b\u200b\u200b \u200b\u200b 2 \u200b3\u200b \u200b4 \u200b5 \u200b6 \u200b7 \u200b\u200b'),
 Document(id='cef5b842-10e8-4443-8372-514ad6cd6313', metadata={'title': '매뉴얼 검토1'}, page_content='매뉴얼 검토1')]

검색된 문서 내용을 컨텍스트로 합쳐서 LLM에 전달합니다.

In [33]:
context = "\n\n".join(
    f"[문서 {i}]\n{doc.page_content}"
    for i, doc in enumerate(retrieved_docs, start=1)
)

print(context[:3000])

[문서 1]
메뉴변경 및 타 사용자 수정 테스트 개정번호 ​ 시행일자 개정내용 작성자 비고 0 20XX. XX. XX 신규 ​ 수정 내용 등 ​ ■ 중분류 업무명 : ■ 소분류 업무명 : 목적 ■ 담당자 : ㅇㅇㅇ / ■ 팀장 : ㅇㅇㅇ NO 업무절차 산출자료/팀명 ​ 입력자료/팀명 자료(서비스)명 수신부서 결재경로 ​ 자료(서비스)명 ​ 발신부서 1 ​​​ ​​ 2 ​3​ ​4 ​5 ​6 ​7 ​​

[문서 2]
메뉴변경 및 타 사용자 수정 테스트 개정번호 ​ 시행일자 개정내용 작성자 비고 0 20XX. XX. XX 신규 ​ 수정 내용 등 ​ ■ 중분류 업무명 : ■ 소분류 업무명 : 목적 ■ 담당자 : ㅇㅇㅇ / ■ 팀장 : ㅇㅇㅇ NO 업무절차 산출자료/팀명 ​ 입력자료/팀명 자료(서비스)명 수신부서 결재경로 ​ 자료(서비스)명 ​ 발신부서 1 ​​​ ​​ 2 ​3​ ​4 ​5 ​6 ​7 ​​

[문서 3]
매뉴얼 검토1


In [34]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model=os.getenv("CHAT_MODEL", "gpt-4o-mini"), temperature=0)

prompt = ChatPromptTemplate.from_template(
    """
    너는 주어진 문서만 근거로 답변하는 assistant다.
    아래 참고 문서를 바탕으로 질문에 답변해라.
    문서에 없는 내용은 추측하지 말고 '문서에서 확인할 수 없습니다.'라고 답변해라.

    [참고 문서]
    {context}

    [질문]
    {question}
    """
)

chain = prompt | llm
response = chain.invoke({"context": context, "question": question})

print(response.content)

문서에서 확인할 수 없습니다.


툴콜링으로 `메뉴얼 총 몇 개` 질문을 처리합니다. LLM이 DB 카운트 툴을 호출하고, 결과를 받아 최종 답변합니다.

In [52]:
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

available_tables = {row[0] for row in tables}

@tool
def count_manuals_tool() -> str:
    """현재 선택한 메뉴얼 테이블의 전체 건수를 반환합니다."""
    if table_name not in available_tables:
        raise ValueError(f"유효하지 않은 테이블명입니다: {table_name}")

    query = text(f"SELECT COUNT(*) AS total FROM `{table_name}`")
    with engine.connect() as conn:
        total = conn.execute(query).scalar_one()

    return str(total)

@tool
def say_hungry_tool() -> str:
    """배고파보이는 문장이 있으면 밥먹어!! 를 반환해 나머지 문장은 답변하지마"""
    return "밥먹어!!!"


@tool
def say_kortek_tool() -> str:
    """코텍이라는 내용이 포함되면 코텍의 기둥 손민욱 대리님에게 직접 물어보세요라고 답변해줘"""
    return "코텍의 기둥 손민욱 대리님에게 직접 물어보세요"

tools = [count_manuals_tool, say_hungry_tool,say_kortek_tool]
tool_map = {tool.name: tool for tool in tools}

def run_tool_call(tool_call: dict) -> str:
    tool_name = tool_call["name"]
    selected_tool = tool_map.get(tool_name)
    if selected_tool is None:
        raise ValueError(f"등록되지 않은 툴입니다: {tool_name}")

    tool_args = tool_call.get("args", {})
    result = selected_tool.invoke(tool_args)
    return str(result)

    


In [53]:
from langchain_openai import ChatOpenAI

tool_llm = ChatOpenAI(model=os.getenv("CHAT_MODEL", "gpt-4o-mini"), temperature=0)
tool_llm = tool_llm.bind_tools(tools)

# tool_question = "현재 메뉴얼이 몇 개 있는지 알려줘."
# tool_question = "배고파 메뉴 추천해줄래?"
tool_question = "코텍 전화번호가 뭐야?"
first_response = tool_llm.invoke([HumanMessage(content=tool_question)])
first_response.tool_calls


[{'name': 'say_kortek_tool',
  'args': {},
  'id': 'call_1cRIvCQ2HXRHPSuW12oLZ3xl',
  'type': 'tool_call'}]

In [55]:
import json
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser


class ApiResponse(BaseModel):
    answer: str = Field(description="사용자에게 보여줄 최종 답변")
    used_tools: list[str] = Field(default_factory=list, description="호출된 툴 이름 목록")
    tool_results: dict[str, str] = Field(default_factory=dict, description="툴 실행 결과")


parser = PydanticOutputParser(pydantic_object=ApiResponse)

messages = [HumanMessage(content=tool_question), first_response]
used_tools: list[str] = []
tool_results: dict[str, str] = {}

for tool_call in first_response.tool_calls:
    tool_name = tool_call["name"]
    try:
        result = run_tool_call(tool_call)
    except Exception as e:
        result = f"tool_error: {e}"

    used_tools.append(tool_name)
    tool_results[tool_name] = str(result)
    messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))

if not first_response.tool_calls:
    final_answer = first_response.content
else:
    final_response = tool_llm.invoke(messages)
    final_answer = final_response.content

structured = parser.parse(
    ApiResponse(
        answer=final_answer,
        used_tools=used_tools,
        tool_results=tool_results,
    ).model_dump_json()
)

# API 응답으로 바로 사용 가능한 JSON
api_json = structured.model_dump()
print(json.dumps(api_json, ensure_ascii=False, indent=2))
api_json


{
  "answer": "코텍의 기둥 손민욱 대리님에게 직접 물어보세요.",
  "used_tools": [
    "say_kortek_tool"
  ],
  "tool_results": {
    "say_kortek_tool": "코텍의 기둥 손민욱 대리님에게 직접 물어보세요"
  }
}


{'answer': '코텍의 기둥 손민욱 대리님에게 직접 물어보세요.',
 'used_tools': ['say_kortek_tool'],
 'tool_results': {'say_kortek_tool': '코텍의 기둥 손민욱 대리님에게 직접 물어보세요'}}